# Homework 5 — Analysing two weeks of plays

About 60 minutes. You analyse **all 14 days** of raw play events (1-14 September) with Spark.

In [3]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: c:\Users\LENOVO\Desktop\Creative Hub\MODUL 1\data-engineering-modul-1\data


In [4]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 3.5.9 is ready


### Exercise 1 — Load and clean
1. Read **all** files `DATA / "raw" / "plays_*.csv"` with the schema below.
2. Remove duplicate events (by `event_id`) and rows with `ms_played < 0`.
3. Add a column `play_date` (`F.to_date("event_ts")`).

Store the result in `plays`.

In [ ]:
PLAYS_SCHEMA = (
    "event_id STRING, event_ts TIMESTAMP, user_id STRING, "
    "track_id STRING, ms_played INT, device STRING, "
    "country STRING, skipped BOOLEAN, shuffle BOOLEAN"
)

plays = (
    spark.read.csv(
        str(DATA / "raw" / "plays_*.csv"),
        header=True,
        schema=PLAYS_SCHEMA
    )
    .dropDuplicates(["event_id"])
    .filter(F.col("ms_played") >= 0)
    .withColumn("play_date", F.to_date("event_ts"))
)

print(
    plays.count(), "plays on",
    plays.select("play_date").distinct().count(), "days"
)

plays.show(5)

### Exercise 2 — Device share
For each device: number of plays and `share` = percent of all plays (rounded to 1 decimal).
Hint: compute `total = plays.count()` first and use it in `withColumn`.

In [ ]:
total = plays.count()

device_share = (
    plays
    .groupBy("device")
    .count()
    .withColumnRenamed("count", "plays")
    .withColumn(
        "share",
        F.round(F.col("plays") / total * 100, 1)
    )
    .orderBy(F.desc("plays"))
)

device_share.show()

### Exercise 3 — When do people listen?
`F.hour(ts)` gives the hour, `F.dayofweek(ts)` gives 1 (Sunday) ... 7 (Saturday).
1. Add `hour` and `is_weekend` (`True` for Saturday and Sunday).
2. Count plays per `is_weekend` and `hour`.
3. Show the 3 busiest hours on weekdays and the 3 busiest on weekends. Are they different?

In [ ]:
# 1. Krijojmë kolonat hour dhe is_weekend
plays_time = (
    plays
    .withColumn("hour", F.hour("event_ts"))
    .withColumn(
        "is_weekend",
        F.dayofweek("event_ts").isin(1, 7)
    )
)

# 2. Numërojmë dëgjimet sipas ditës dhe orës
hourly_plays = (
    plays_time
    .groupBy("is_weekend", "hour")
    .count()
)

# 3. Top 3 orët gjatë ditëve të javës
print("Weekdays:")

hourly_plays \
    .filter(F.col("is_weekend") == False) \
    .orderBy(F.desc("count")) \
    .show(3)

# 4. Top 3 orët gjatë fundjavës
print("Weekends:")

hourly_plays \
    .filter(F.col("is_weekend") == True) \
    .orderBy(F.desc("count")) \
    .show(3)

### Exercise 4 — Does shuffle make people skip?
Compute the skip rate (percent, 1 decimal) for `shuffle = true` and `shuffle = false`.

In [ ]:
skip_by_shuffle = (
    plays
    .groupBy("shuffle")
    .agg(
        F.count("*").alias("plays"),
        F.round(
            F.avg(F.col("skipped").cast("int")) * 100, 1
        ).alias("skip_rate")
    )
    .orderBy("shuffle")
)

skip_by_shuffle.show()

### Exercise 5 — Explicit content by plan
Load `tracks.csv` and `users.csv`. For each `plan`, what percentage of plays are of **explicit** tracks?

In [ ]:
# 1. Lexojmë tabelat
tracks = spark.read.csv(
    str(DATA / "reference" / "tracks.csv"),
    header=True,
    inferSchema=True
)

users = spark.read.csv(
    str(DATA / "reference" / "users.csv"),
    header=True,
    inferSchema=True
)

# 2. Bashkojmë të dhënat
plays_joined = (
    plays
    .join(
        tracks.select("track_id", "explicit"),
        on="track_id",
        how="inner"
    )
    .join(
        users.select("user_id", "plan"),
        on="user_id",
        how="inner"
    )
)

# 3. Llogarisim përqindjen
explicit_by_plan = (
    plays_joined
    .groupBy("plan")
    .agg(
        F.count("*").alias("plays"),
        F.round(
            F.avg(F.col("explicit").cast("int")) * 100,
            1
        ).alias("explicit_percent")
    )
)

explicit_by_plan.show()

### Exercise 6 — Save a result table
1. Build `daily_country`: per `play_date` and `country`: `plays` and `hours` (rounded to 2 decimals).
2. Write it as Parquet to `DATA.parent / "output" / "daily_country"`, **partitioned by country**, with mode `overwrite`.
3. Read back only Austria (`country=AT` sub-folder or filter after reading) and show it sorted by date.

In [ ]:
# 1. Krijojmë tabelën daily_country
daily_country = (
    plays
    .groupBy("play_date", "country")
    .agg(
        F.count("*").alias("plays"),
        F.round(
            F.sum("ms_played") / 3600000,
            2
        ).alias("hours")
    )
    .orderBy("play_date", "country")
)

daily_country.show(10)

# 2. Ruajmë në Parquet
output_path = DATA.parent / "output" / "daily_country"

(
    daily_country.write
    .mode("overwrite")
    .partitionBy("country")
    .parquet(str(output_path))
)

# 3. Lexojmë rezultatin vetëm për Austrinë
austria = (
    spark.read.parquet(str(output_path))
    .filter(F.col("country") == "AT")
    .orderBy("play_date")
)

austria.show()

### Exercise 7 (Stretch) — Top 3 tracks per genre
Show, for every genre, the 3 most played tracks with their title and artist name. Use a window function with `F.row_number()`.

In [ ]:
from pyspark.sql.window import Window

# 1. Lexojmë artistët
artists = spark.read.csv(
    str(DATA / "reference" / "artists.csv"),
    header=True,
    inferSchema=True
)

# 2. Bashkojmë plays, tracks dhe artists
plays_details = (
    plays
    .join(
        tracks.select("track_id", "title", "genre", "artist_id"),
        on="track_id",
        how="inner"
    )
    .join(
        artists.select("artist_id", "artist_name"),
        on="artist_id",
        how="inner"
    )
)

# 3. Numërojmë dëgjimet për çdo këngë
track_counts = (
    plays_details
    .groupBy("genre", "track_id", "title", "artist_name")
    .agg(
        F.count("*").alias("plays")
    )
)

# 4. Krijojmë Window për renditje brenda çdo zhanri
window_spec = (
    Window
    .partitionBy("genre")
    .orderBy(F.desc("plays"), "track_id")
)

# 5. Marrim 3 këngët më të dëgjuara për çdo zhanër
top_3_tracks = (
    track_counts
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 3)
    .orderBy("genre", "rank")
)

top_3_tracks.show(100, truncate=False)